### Imports

In [1]:
import random as r
import numpy as np
r.seed(1)

### Unit Tests

In [2]:
#Tolerance is used to handle floating point rounding
def is_valid_row(x, tolerance = 10 ** -5):
    assert(1 - tolerance < sum(x) < 1 + tolerance)
    
def is_valid_probability_matrix(x):
    for xprime in x:
        is_valid_row(xprime)

### Transition Matrix

In [3]:
transition_matrix = np.array([
    [0.10, 0.60, 0.20, 0.10],
    [0.30, 0.20, 0.40, 0.10],
    [0.25, 0.25, 0.25, 0.25],
    [0.00, 0.10, 0.30, 0.60],
])

is_valid_probability_matrix(transition_matrix)

### Multi-Step Transition Matrix

Using matrix multiplication we can work out $P^{2}$ etc

In [4]:
def multi_step_transition(transition_matrix, num_steps):
    output = [c for c in transition_matrix]
    for i in range(num_steps - 1):
        output = output @ transition_matrix
        is_valid_probability_matrix(output)
    return(output)

for i in range(9):
    multi_step_transition_matrix = multi_step_transition(transition_matrix, i + 2)
    print(f"P{i + 2} {multi_step_transition_matrix}")
    print("")

P2 [[0.24   0.24   0.34   0.18  ]
 [0.19   0.33   0.27   0.21  ]
 [0.1625 0.2875 0.2875 0.2625]
 [0.105  0.155  0.295  0.445 ]]

P3 [[0.181    0.295    0.283    0.241   ]
 [0.1855   0.2685   0.3005   0.2455  ]
 [0.174375 0.253125 0.298125 0.274375]
 [0.13075  0.21225  0.29025  0.36675 ]]

P4 [[0.17735    0.26245    0.29725    0.26295   ]
 [0.174225   0.264675   0.293275   0.267825  ]
 [0.16790625 0.25721875 0.29296875 0.28190625]
 [0.1493125  0.2301375  0.2936375  0.3269125 ]]

P5 [[0.1707825  0.2595075  0.2936475  0.2760625 ]
 [0.17014375 0.25757125 0.29438125 0.27790375]
 [0.16719844 0.25362031 0.29428281 0.28489844]
 [0.15738188 0.24171563 0.29340063 0.30750187]]

P6 [[0.16834237 0.25538912 0.29419013 0.28207837]
 [0.16788106 0.25498619 0.29402369 0.28310906]
 [0.16637664 0.25310367 0.29392805 0.28659164]
 [0.16160303 0.24687259 0.29376334 0.29776103]]

P7 [[0.16699851 0.25383862 0.29399517 0.28516771]
 [0.16678988 0.2535427  0.29400933 0.28565808]
 [0.16605078 0.25258789 0.2939763 

# Chapman-Kolmogorov

Chapman-Kolmogorov tells us that:

$$P_{ij}^{m + n} = \Sigma_{k} P_{ik}^{m} \cdot P_{kj}^{n}$$

Let's validate this! If it's correct: $$P^{5} = P^{2} \cdot P^{3}$$ so let's check that!

In [5]:
tolerance = 10 ** -5
P5a = multi_step_transition(transition_matrix, 5)
P5b = multi_step_transition(transition_matrix, 2) @ multi_step_transition(transition_matrix, 3)
assert(sum(sum(P5a - P5b)) < tolerance)

### Empirical Verification

Let's run $10^{5}$ simulations of a 5-step process and approximate the empirical $P^{5}$, then we'll compare to the theoretical $P^{5}$ and see whether our predictions match observation.

In [6]:
simulation_tolerance = 0.1
num_trials = 10 ** 5
empirical_P5 = [[0 for i in range(4)] for j in range(4)]

for j in range(num_trials):
    initial_state = r.randint(0, 3)
    state = initial_state
    for i in range(5):
        state = np.random.choice(a = 4,p = transition_matrix[state])
    empirical_P5[initial_state][state] += 1
    
empirical_P5 = np.array([[i * 4 / num_trials for i in empirical_P5[j]] for j in range(len(empirical_P5))])
    
    
print(f"Empirical : {empirical_P5}")
print("")
print(f"Predicted: {P5a}")
print(f"Maximum empirical error: {np.max(np.abs(P5a - empirical_P5)):.5f}")
assert(sum(sum(P5a - empirical_P5)) < simulation_tolerance)

Empirical : [[0.17452 0.2594  0.28984 0.27524]
 [0.17344 0.25644 0.29328 0.27872]
 [0.16196 0.25164 0.29048 0.28516]
 [0.15704 0.24388 0.2958  0.31316]]

Predicted: [[0.1707825  0.2595075  0.2936475  0.2760625 ]
 [0.17014375 0.25757125 0.29438125 0.27790375]
 [0.16719844 0.25362031 0.29428281 0.28489844]
 [0.15738188 0.24171563 0.29340063 0.30750187]]
Maximum empirical error: 0.00566
